# Music Annotation Project 59R4A
## Targeted Adjudication of Page/Scan Edges, Region Fragments and Model Disagreements

**Runtime:** CPU. Run after Notebook 59R4.

This notebook does not retrain a model. It preserves the original Notebook 66 decisions and creates a separate `review_adjudications.csv` containing only explicit later corrections.

The review queue is deliberately small and targeted:
- all human `UNCERTAIN` cases;
- apparent page/scan-edge candidates;
- multiscale regressions against the scalar/local33 arms;
- high-disagreement cases;
- visibly incomplete region fragments.

The reviewer records one required outcome:

`ANNOTATION / PRINT / MIXED / PAGE_SCAN_EDGE / UNCERTAIN / REGION_FRAGMENT / MASK_ERROR`

`REGION_FRAGMENT` is an exclusion/status for a semantically incomplete candidate region, not a correction class.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q gradio pandas numpy opencv-python-headless pillow


Mounted at /content/drive


In [7]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile
import numpy as np
import pandas as pd
import cv2
import gradio as gr
from PIL import Image

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
R59R4_ROOT = ROOT / 'notebook59R4_multiscale_region_semantics'
OUTPUT_ROOT = ROOT / 'region_semantic_adjudication_59R4A'

SOURCE_59R4_RUN_OVERRIDE = None
DECISIONS_CSV_OVERRIDE = None
MAX_QUEUE = 60
PUBLIC_LINK = True
BORDER_MARGIN_FRACTION = 0.025
DISAGREEMENT_MODELS = ['pred_scalar','pred_local33','pred_multiscale33_97_257']

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_59R4A_targeted_adjudication'
OUT = OUTPUT_ROOT / RUN
TABLES = OUT/'tables'; REVIEW = OUT/'review'; AUDIT = OUT/'audit'; UPLOAD = OUT/'upload'
for p in [OUT,TABLES,REVIEW,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

def require(ok,msg):
    if not ok: raise RuntimeError(msg)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def complete_run(path):
    return all((path/x).exists() for x in [
        'tables/out_of_fold_predictions.csv','tables/model_summary.csv','audit/notebook59R4_verdict.json'
    ])

def latest_complete(root):
    hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and complete_run(p): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def find_unique_below(root,name,label,required=True):
    hits=sorted({p.resolve() for p in Path(root).rglob(name) if p.is_file()})
    if not hits and not required: return None
    require(len(hits)==1,f'{label}: expected one {name} below {root}, found {len(hits)}: {hits[:20]}')
    return hits[0]

SOURCE = Path(SOURCE_59R4_RUN_OVERRIDE) if SOURCE_59R4_RUN_OVERRIDE else latest_complete(R59R4_ROOT)
require(SOURCE is not None and complete_run(SOURCE),'No complete Notebook 59R4 run found. Set SOURCE_59R4_RUN_OVERRIDE.')
PRED_PATH = SOURCE/'tables'/'out_of_fold_predictions.csv'
DECISIONS_PATH = Path(DECISIONS_CSV_OVERRIDE) if DECISIONS_CSV_OVERRIDE else find_unique_below(SOURCE,'notebook66_completed_snapshot.csv','Notebook 66 decisions')
print('59R4:',SOURCE)
print('Decisions:',DECISIONS_PATH)
print('Output:',OUT)


59R4: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4
Decisions: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4/inputs/notebook66_completed_snapshot.csv
Output: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_adjudication_59R4A/20261115_091326_59R4A_targeted_adjudication


In [8]:
pred=pd.read_csv(PRED_PATH,dtype=str).fillna('')
dec=pd.read_csv(DECISIONS_PATH,dtype=str).fillna('')
required={'region_id','page','semantic_label','annotation_family'}
require(required.issubset(dec.columns),f'review_decisions.csv missing {sorted(required-set(dec.columns))}')
require('region_id' in pred.columns,'59R4 predictions lack region_id')

# Preserve the original review as immutable evidence.
shutil.copy2(DECISIONS_PATH,AUDIT/'review_decisions_original.csv')
shutil.copy2(PRED_PATH,AUDIT/'out_of_fold_predictions_original.csv')

work=dec.merge(pred,on='region_id',how='left',validate='one_to_one',suffixes=('','_59R4'))
require(work['page'].ne('').all(),'Missing page identity after merge')

# Normalise model prediction aliases.
def first_col(frame,names):
    return next((c for c in names if c in frame.columns),None)
model_cols={
 'scalar':first_col(work,['pred_scalar','scalar_prediction','scalar_pred']),
 'local33':first_col(work,['pred_local33','local33_prediction','local33_pred']),
 'multi':first_col(work,['pred_multiscale33_97_257','pred_multiscale','multiscale_prediction','multiscale_pred']),
}
require(all(model_cols.values()),f'Could not resolve model prediction columns: {model_cols}')
for k,c in model_cols.items(): work[f'_pred_{k}']=work[c].astype(str).str.upper().str.strip()

# Geometry aliases used only for queue prioritisation, never as labels.
def num(names,default=np.nan):
    c=first_col(work,names)
    return pd.to_numeric(work[c],errors='coerce') if c else pd.Series(default,index=work.index,dtype=float)
work['_x0']=num(['x0','bbox_x0','x','left'])
work['_y0']=num(['y0','bbox_y0','y','top'])
work['_x1']=num(['x1','bbox_x1','right'])
work['_y1']=num(['y1','bbox_y1','bottom'])
work['_page_w']=num(['page_width','image_width','page_w'])
work['_page_h']=num(['page_height','image_height','page_h'])
if work['_x1'].isna().all(): work['_x1']=work['_x0']+num(['bbox_w','w','width'])
if work['_y1'].isna().all(): work['_y1']=work['_y0']+num(['bbox_h','h','height'])

border=(
 (work['_x0']<=BORDER_MARGIN_FRACTION*work['_page_w']) |
 (work['_y0']<=BORDER_MARGIN_FRACTION*work['_page_h']) |
 (work['_x1']>=(1-BORDER_MARGIN_FRACTION)*work['_page_w']) |
 (work['_y1']>=(1-BORDER_MARGIN_FRACTION)*work['_page_h'])
).fillna(False)
uncertain=work.semantic_label.str.upper().eq('UNCERTAIN')
scalar_reg=(work['_pred_scalar'].eq(work.semantic_label.str.upper()) & ~work['_pred_multi'].eq(work.semantic_label.str.upper()))
local_reg=(work['_pred_local33'].eq(work.semantic_label.str.upper()) & ~work['_pred_multi'].eq(work.semantic_label.str.upper()))
disagree=(work[['_pred_scalar','_pred_local33','_pred_multi']].nunique(axis=1)>1)

work['_queue_reason']=''
for mask,label in [(uncertain,'HUMAN_UNCERTAIN'),(border,'BORDER_CANDIDATE'),(scalar_reg,'MULTISCALE_REGRESSION_VS_SCALAR'),(local_reg,'MULTISCALE_REGRESSION_VS_LOCAL33'),(disagree,'MODEL_DISAGREEMENT')]:
    work.loc[mask,'_queue_reason']=work.loc[mask,'_queue_reason'].map(lambda x:'|'.join(filter(None,[x,label])))
queue=work[work['_queue_reason'].ne('')].copy()
queue['_priority']=uncertain.loc[queue.index].astype(int)*100 + border.loc[queue.index].astype(int)*50 + scalar_reg.loc[queue.index].astype(int)*20 + local_reg.loc[queue.index].astype(int)*10 + disagree.loc[queue.index].astype(int)*5
queue=queue.sort_values(['_priority','page'],ascending=[False,True]).drop_duplicates('region_id')
if MAX_QUEUE is not None: queue=queue.head(MAX_QUEUE)
require(len(queue)>0,'Targeted 59R4A queue is empty.')
queue['review_order']=np.arange(1,len(queue)+1)
queue.to_csv(TABLES/'adjudication_queue.csv',index=False)
print('Queue:',len(queue),'regions across',queue.page.nunique(),'pages')
print(queue._queue_reason.value_counts().head(20))


RuntimeError: Could not resolve model prediction columns: {'scalar': 'scalar_prediction', 'local33': 'local33_prediction', 'multi': None}

In [ ]:
# Resolve visual paths from 59R4 output columns or common folders.
IMAGE_COLUMNS=['context_257_path','context_path','review_image_path','crop_path','image_path','observed_path']
def resolve_image(row):
    for c in IMAGE_COLUMNS:
        if c in row.index and str(row[c]).strip():
            p=Path(str(row[c]))
            if p.exists(): return p
            q=SOURCE/p
            if q.exists(): return q
    # Prefer pre-rendered region files carrying region_id.
    rid=str(row.region_id)
    hits=[p for p in SOURCE.rglob('*') if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg'} and rid in p.stem]
    if hits: return sorted(hits,key=lambda p:p.stat().st_mtime)[-1]
    # Fallback to page-level observed images.
    page=str(row.page)
    candidates=[]
    for pat in [f'{page}.png',f'{page}.jpg',f'{page}.jpeg','observed.png']:
        candidates.extend(SOURCE.rglob(pat))
    candidates=[p for p in candidates if page in str(p) or p.name=='observed.png']
    return sorted(candidates,key=lambda p:p.stat().st_mtime)[-1] if candidates else None

queue['_resolved_image']=queue.apply(resolve_image,axis=1)
missing=queue[queue['_resolved_image'].isna()][['region_id','page']]
missing.to_csv(AUDIT/'missing_review_images.csv',index=False)
require(len(missing)==0,f'{len(missing)} queue rows have no resolvable image. See missing_review_images.csv')
queue.to_csv(TABLES/'adjudication_queue_resolved.csv',index=False)

def render(row,margin=160):
    p=Path(row['_resolved_image']); im=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
    require(im is not None,f'Unreadable image {p}')
    x0=row['_x0'];y0=row['_y0'];x1=row['_x1'];y1=row['_y1']
    if all(np.isfinite([x0,y0,x1,y1])) and x1>x0 and y1>y0 and im.shape[1]>=x1 and im.shape[0]>=y1:
        xa=max(0,int(x0)-margin); ya=max(0,int(y0)-margin); xb=min(im.shape[1],int(x1)+margin); yb=min(im.shape[0],int(y1)+margin)
        crop=cv2.cvtColor(im[ya:yb,xa:xb],cv2.COLOR_GRAY2RGB)
        cv2.rectangle(crop,(int(x0)-xa,int(y0)-ya),(int(x1)-xa,int(y1)-ya),(255,40,40),3)
        return crop
    return cv2.cvtColor(im,cv2.COLOR_GRAY2RGB)
print('Images resolved.')


In [ ]:
ADJ_PATH=TABLES/'review_adjudications.csv'
LABELS=['ANNOTATION','PRINT','MIXED','PAGE_SCAN_EDGE','UNCERTAIN','REGION_FRAGMENT','MASK_ERROR']
STATE={'position':0}

def load_existing():
    if ADJ_PATH.exists(): return pd.read_csv(ADJ_PATH,dtype=str).fillna('')
    return pd.DataFrame(columns=['region_id','adjudicated_semantic_label','adjudicated_annotation_family','reviewer','reason','adjudicated_utc','original_semantic_label','original_annotation_family','queue_reason'])

def current_row():
    return queue.iloc[min(STATE['position'],len(queue)-1)]

def show():
    r=current_row(); done=len(load_existing())
    title=f"{STATE['position']+1}/{len(queue)} | {r.page} | original={r.semantic_label} | {r._queue_reason}"
    return render(r),title,gr.update(value=None),f'Completed {done}/{len(queue)}'

def save(label,reason,reviewer):
    require(label in LABELS,'Choose one adjudication label.')
    r=current_row(); old=load_existing(); now=datetime.now(timezone.utc).isoformat()
    row={'region_id':r.region_id,'adjudicated_semantic_label':label,'adjudicated_annotation_family':'NOT_APPLICABLE' if label not in ['ANNOTATION','MIXED'] else r.annotation_family,'reviewer':reviewer.strip() or 'David Bolton','reason':reason.strip(),'adjudicated_utc':now,'original_semantic_label':r.semantic_label,'original_annotation_family':r.annotation_family,'queue_reason':r._queue_reason}
    old=old[old.region_id!=r.region_id]
    pd.concat([old,pd.DataFrame([row])],ignore_index=True).to_csv(ADJ_PATH,index=False)
    STATE['position']=min(STATE['position']+1,len(queue)-1)
    return show()

def previous():
    STATE['position']=max(0,STATE['position']-1); return show()

def next_without_save():
    STATE['position']=min(len(queue)-1,STATE['position']+1); return show()

with gr.Blocks(title='Notebook 59R4A Targeted Adjudication') as app:
    gr.Markdown('# 59R4A Targeted Adjudication')
    status=gr.Textbox(label='Candidate',interactive=False)
    image=gr.Image(label='Context with candidate box',height=720)
    label=gr.Radio(LABELS,label='Correct semantic outcome')
    reason=gr.Textbox(label='Optional adjudication reason')
    reviewer=gr.Textbox(value='David Bolton',label='Reviewer')
    progress=gr.Textbox(label='Progress',interactive=False)
    with gr.Row():
        prev=gr.Button('Previous'); save_next=gr.Button('Save + next',variant='primary'); skip=gr.Button('Next without saving')
    app.load(show,outputs=[image,status,label,progress])
    save_next.click(save,[label,reason,reviewer],[image,status,label,progress])
    prev.click(previous,outputs=[image,status,label,progress])
    skip.click(next_without_save,outputs=[image,status,label,progress])
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)


In [ ]:
# Run after closing/completing the reviewer.
adj=load_existing()
summary=(adj.groupby(['queue_reason','adjudicated_semantic_label'],as_index=False).size().rename(columns={'size':'regions'})) if len(adj) else pd.DataFrame()
summary.to_csv(TABLES/'adjudication_summary.csv',index=False)
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'59R4A','source_59R4_run':str(SOURCE),'source_predictions_sha256':sha256_file(PRED_PATH),'source_decisions_sha256':sha256_file(DECISIONS_PATH),'queue_rows':len(queue),'queue_pages':int(queue.page.nunique()),'adjudicated_rows':len(adj),'original_decisions_modified':False,'adjudication_file':str(ADJ_PATH) if ADJ_PATH.exists() else None}
atomic_json(OUT/'notebook59R4A_verdict.json',record)
(OUT/'executive_summary.txt').write_text(chr(10).join(['NOTEBOOK 59R4A TARGETED ADJUDICATION','',f'Queue: {len(queue)} regions across {queue.page.nunique()} pages',f'Adjudicated: {len(adj)}',f'Adjudication file: {ADJ_PATH}','Original Notebook 66 decisions modified: False']))

# Curated upload package.
files=[OUT/'executive_summary.txt',OUT/'notebook59R4A_verdict.json',TABLES/'adjudication_queue.csv',TABLES/'adjudication_queue_resolved.csv']
if ADJ_PATH.exists(): files.append(ADJ_PATH)
if (TABLES/'adjudication_summary.csv').exists(): files.append(TABLES/'adjudication_summary.csv')
rows=[]
for p in files:
    require(p.exists(),f'Missing upload artefact {p}')
    dst=UPLOAD/p.name; shutil.copy2(p,dst); rows.append({'file':p.name,'sha256':sha256_file(dst),'bytes':dst.stat().st_size})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_path=OUT/'notebook59R4A_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir(): z.write(p,p.name)
print('59R4A complete:',OUT)
print('Adjudications:',ADJ_PATH)
print('Upload bundle:',zip_path)
